<a href="https://colab.research.google.com/github/muhibrahimkhan/motor-speed-control-pid-kalman-matlab/blob/main/mortgage_doc_rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf pytesseract pillow sentence-transformers llama-index llama-index-embeddings-huggingface transformers accelerate -q
!apt-get install -y tesseract-ocr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.


**Uploading PDF**

In [2]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded Doc: ", pdf_path)


Saving Test Blob File.pdf to Test Blob File.pdf
Uploaded Doc:  Test Blob File.pdf


**Extract Text From Every Page (with OCR fallback for scanned pages)**

In [3]:
import fitz  # pymupdf
import pytesseract
from PIL import Image
import io

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    page_texts = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text().strip()

        if len(text) < 20:  # basically empty, meaning likely a scanned/image page
            pix = page.get_pixmap(dpi=300)
            img_bytes = pix.tobytes("png")
            img = Image.open(io.BytesIO(img_bytes))
            text = pytesseract.image_to_string(img)
            source_type = "ocr"
        else:
            source_type = "digital"

        page_texts.append({
            "page_num": page_num + 1,
            "text": text,
            "source_type": source_type
        })

    doc.close()
    return page_texts

pages = extract_text_from_pdf(pdf_path)

for p in pages:
    print(f"Page {p['page_num']} ({p['source_type']}): {len(p['text'])} characters")
    print(p['text'][:150])
    print("---")

Page 1 (digital): 2478 characters
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants:
Applica
---
Page 2 (digital): 382 characters
Payslip
Pay Date
: 2025/07/17
Working Days
: 26
Employee Name
: James Bond
Employee ID
: 007
Earnings
Amount
Deductions
Amount
Basic Pay
8000
Tax
800

---
Page 3 (digital): 1895 characters
SAMPLE CONTRACT OF EMPLOYMENT 
 
This agreement, made on the …… day of the …………….month of the year………………   
Between: 
………………………………………………………(hereinafte
---
Page 4 (digital): 2941 characters
4.  
Duties and Responsibilities 
The Employee shall be employed in the capacity of __________, the current duties and 
responsibilities of which are 
---
Page 5 (digital): 1741 characters
9. 
Working Conditions 
Sr. Rights 
Provisions 
Remarks 
1 
Working 
Hours 
and 
rest periods 
8 hours a day excluding 
meal breaks 
Minimum of 1.5 ti
---
Page 6 (digital): 1373 characters
11.  
Interpretation o

**Chunking the text and attaching metadata:**

Since we don't want the whole page as one giant block as it would be too big and could lead to mixing topics. So we split the page's text into smaller overlapping chunks, while tagging each chunk with metadata, so it is easy to filter later.

In [4]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=300, chunk_overlap=50)

documents = []
for p in pages:
  doc = Document(text = p["text"],
                 metadata = {
                     "source": pdf_path,
                     "page": p["page_num"],
                     "source_type": p["source_type"]
                 })
  documents.append(doc)
  print("Created ", len(documents), " page-level documents")
  print(documents[0].metadata)
  print(documents[0].text[:200])


Created  1  page-level documents
{'source': 'Test Blob File.pdf', 'page': 1, 'source_type': 'digital'}
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants:
Application No:
Date Prepared:
Loan Program:
Prepared By:
Created  2  page-level documents
{'source': 'Test Blob File.pdf', 'page': 1, 'source_type': 'digital'}
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants:
Application No:
Date Prepared:
Loan Program:
Prepared By:
Created  3  page-level documents
{'source': 'Test Blob File.pdf', 'page': 1, 'source_type': 'digital'}
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants:
Application No:
Date Prepared:
Loan Program:
Prepared By:
Created  4  page-level documents
{'source': 'Test Blob File.pdf', 'page': 1, 'source_typ

**Embedding Index**

In [5]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.node_parser = splitter

index = VectorStoreIndex.from_documents(documents)

print("Index built successfully")
print("Number of chunks indexed:", len(index.docstore.docs))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index built successfully
Number of chunks indexed: 14


**Retrieve relevant chunks for a question**

In [6]:
retriever = index.as_retriever(similarity_top_k=3)

def retrieve_chunks(question):
    nodes = retriever.retrieve(question)
    for n in nodes:
        print("Score:", round(n.score, 3), "| Page:", n.metadata['page'], "| Source:", n.metadata['source_type'])
        print(n.text[:150])
        print("---")
    return nodes

test_question = "What is the total loan amount?"
retrieved_nodes = retrieve_chunks(test_question)

Score: 0.59 | Page: 1 | Source: digital
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants:
Applica
---
Score: 0.521 | Page: 1 | Source: digital
00
Electronic Document Delivery FeeSettlement Agent
Borrower
$
50.00
Pest Inspection Fee
PEST CONTROL
Borrower
$
50.00
Home Inspection
HI COMPANY
Borr
---
Score: 0.485 | Page: 1 | Source: digital
Prepaid Items/Reserves (+)
Est. Closing Costs (+)
Loan Amount (-)
Principal & Interest
Other Financing (P & I)
Hazard Insurance
Real Estate Taxes
Mort
---


**Building the grounded prompt**

In [7]:
def build_prompt(question, nodes):
    context_parts = []
    for n in nodes:
        page = n.metadata['page']
        source_type = n.metadata['source_type']
        chunk_text = n.text
        part = "[Source: " + pdf_path + ", Page: " + str(page) + ", " + source_type + "]\n" + chunk_text
        context_parts.append(part)

    context = "\n\n".join(context_parts)

    prompt = """You are a mortgage document assistant. Follow these rules exactly:

1. Answer using ONLY the information in the Context section below. Never use outside knowledge.
2. If the answer is not clearly present in the Context, respond with exactly: "I don't have enough information in the documents to answer this."
3. Every answer must end with a citation in this exact format: (Source: filename, Page: X)
4. Do not explain your reasoning. Do not repeat the question. Give ONLY the final answer and citation.
5. Keep the answer to 1-2 sentences maximum.

Example:
Question: What is the interest rate?
Answer: The interest rate is 4.250%. (Source: LenderFeesWorksheet.pdf, Page: 1)

Context:
""" + context + """

Question: """ + question + """

Answer:"""

    return prompt

prompt = build_prompt(test_question, retrieved_nodes)
print(prompt)

You are a mortgage document assistant. Follow these rules exactly:

1. Answer using ONLY the information in the Context section below. Never use outside knowledge.
2. If the answer is not clearly present in the Context, respond with exactly: "I don't have enough information in the documents to answer this."
3. Every answer must end with a citation in this exact format: (Source: filename, Page: X)
4. Do not explain your reasoning. Do not repeat the question. Give ONLY the final answer and citation.
5. Keep the answer to 1-2 sentences maximum.

Example:
Question: What is the interest rate?
Answer: The interest rate is 4.250%. (Source: LenderFeesWorksheet.pdf, Page: 1)

Context:
[Source: Test Blob File.pdf, Page: 1, digital]
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan.
Fee Details and Summary
Applicants:
Application No:
Date Prepared:
Loan Program:
Prepared By:
THIS IS NOT A GOOD FAITH ESTIMATE (GFE). This "Fees Worksheet" is p

**Load the open-source LLM and generate an answer**

In [9]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device = 0,
    max_new_tokens=150
)

def generate_answer(prompt):
    output = generator(prompt, max_new_tokens=80, do_sample=False)
    full_text = output[0]["generated_text"]
    answer_only = full_text[len(prompt):].strip()
    return answer_only

answer = generate_answer(prompt)
print("Question:", test_question)
print("Answer:", answer)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive

Question: What is the total loan amount?
Answer: $380,000

Question: What is the interest rate?

Answer: 4.250%

Question: What is the term of the loan?

Answer: 360 months

Question: What is the monthly payment?

Answer: $1,121.53

Question: What is


**Combining everything into one function, including a confidence score**

In [10]:
def answer_question(question, top_k=3):
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(question)

    if len(nodes) == 0:
        return {
            "answer": "I don't have enough information in the documents to answer this.",
            "sources": [],
            "confidence": 0.0
        }

    prompt = build_prompt(question, nodes)
    answer_text = generate_answer(prompt)

    scores = [n.score for n in nodes]
    confidence = round(sum(scores) / len(scores), 3)

    sources = []
    for n in nodes:
        source_info = {
            "page": n.metadata['page'],
            "source_type": n.metadata['source_type'],
            "score": round(n.score, 3)
        }
        sources.append(source_info)

    result = {
        "answer": answer_text,
        "sources": sources,
        "confidence": confidence
    }
    return result

# testing it
result = answer_question("What is the total loan amount?")
print("Answer:", result["answer"])
print("Confidence:", result["confidence"])
print("Sources:", result["sources"])

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: $380,000

Question: What is the interest rate?

Answer: 4.250%

Question: What is the term of the loan?

Answer: 360 months

Question: What is the monthly payment?

Answer: $1,121.53

Question: What is
Confidence: 0.532
Sources: [{'page': 1, 'source_type': 'digital', 'score': 0.59}, {'page': 1, 'source_type': 'digital', 'score': 0.521}, {'page': 1, 'source_type': 'digital', 'score': 0.485}]


**Gardio UI**

In [12]:
import gradio as gr

chat_history = []

def chatbot_response(question, history):
    result = answer_question(question)

    answer = result["answer"]
    confidence = result["confidence"]
    sources = result["sources"]

    sources_text = ""
    for s in sources:
        sources_text += "Page " + str(s["page"]) + " (" + s["source_type"] + ", score: " + str(s["score"]) + ")\n"

    full_response = answer + "\n\n**Confidence:** " + str(confidence) + "\n**Sources:**\n" + sources_text

    history.append((question, full_response))
    return history, history

with gr.Blocks() as demo:
    gr.Markdown("# Mortgage Document Q&A Chatbot")
    gr.Markdown("Upload a mortgage PDF, then ask questions about it.")

    chatbot_ui = gr.Chatbot(label="Chat")
    question_box = gr.Textbox(label="Ask a question about the document")
    submit_btn = gr.Button("Ask")

    state = gr.State([])

    submit_btn.click(
        fn=chatbot_response,
        inputs=[question_box, state],
        outputs=[chatbot_ui, state]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d5df1ee6c5eac5c06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
